In [3]:
# ============================================================
# STAGE 6 - LOAD DASHBOARD DATA
# ============================================================

import pandas as pd
import numpy as np
import os

print("=" * 70)
print("STAGE 6 - DASHBOARD DATA PREPARATION")
print("=" * 70)


# ------------------------------------------------------------
# 1. LOAD PROFITABILITY DATA
# ------------------------------------------------------------

# CHANGE THIS PATH if your profitability file is stored elsewhere
profitability_path = "profitability.csv"

if not os.path.exists(profitability_path):
    raise FileNotFoundError(
        f"Could not find '{profitability_path}'. "
        "Please enter the correct path to your profitability CSV."
    )

profitability = pd.read_csv(profitability_path)

print("\nProfitability dataset loaded successfully.")
print("Shape:", profitability.shape)

print("\nColumns:")
print(profitability.columns.tolist())


# ------------------------------------------------------------
# 2. CHECK REQUIRED COLUMNS
# ------------------------------------------------------------

required_columns = [
    "product_name",
    "revenue",
    "waste_cost",
    "waste_pct",
    "adjusted_profit",
    "adjusted_margin_pct"
]

missing_columns = [
    col for col in required_columns
    if col not in profitability.columns
]

if missing_columns:
    raise ValueError(
        f"\nMissing required columns: {missing_columns}\n"
        f"Available columns: {profitability.columns.tolist()}"
    )

print("\n✓ All required columns are present.")


# ------------------------------------------------------------
# 3. CREATE KPI SUMMARY
# ------------------------------------------------------------

total_revenue = profitability["revenue"].sum()
total_profit = profitability["adjusted_profit"].sum()
total_waste_cost = profitability["waste_cost"].sum()

if "units_sold" in profitability.columns:
    total_units = profitability["units_sold"].sum()
else:
    total_units = np.nan

overall_margin = (
    total_profit / total_revenue * 100
    if total_revenue != 0 else 0
)


print("\n" + "=" * 70)
print("EXECUTIVE KPIs")
print("=" * 70)

print(f"Total Revenue:    {total_revenue:,.2f}")
print(f"Adjusted Profit:  {total_profit:,.2f}")
print(f"Waste Cost:       {total_waste_cost:,.2f}")
print(f"Units Sold:       {total_units:,.0f}")
print(f"Profit Margin:    {overall_margin:.2f}%")


# ------------------------------------------------------------
# 4. CREATE PRODUCT DASHBOARD DATA
# ------------------------------------------------------------

dashboard_products = profitability.copy()


# Profit ranking
dashboard_products["profit_rank"] = (
    dashboard_products["adjusted_profit"]
    .rank(method="min", ascending=False)
    .astype(int)
)


# Waste ranking
dashboard_products["waste_rank"] = (
    dashboard_products["waste_cost"]
    .rank(method="min", ascending=False)
    .astype(int)
)


# Margin ranking
dashboard_products["margin_rank"] = (
    dashboard_products["adjusted_margin_pct"]
    .rank(method="min", ascending=False)
    .astype(int)
)


# ------------------------------------------------------------
# 5. CLASSIFY PRODUCTS
# ------------------------------------------------------------

def classify_product(row):

    margin = row["adjusted_margin_pct"]
    waste = row["waste_pct"]

    if margin < 5 and waste >= 20:
        return "Critical"

    elif margin < 10:
        return "Low Margin"

    elif waste >= 20:
        return "High Waste"

    elif margin >= 25:
        return "Strong"

    else:
        return "Moderate"


dashboard_products["business_status"] = (
    dashboard_products.apply(classify_product, axis=1)
)


# ------------------------------------------------------------
# 6. DISPLAY RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PRODUCT PERFORMANCE")
print("=" * 70)

display(
    dashboard_products.sort_values(
        "adjusted_profit",
        ascending=False
    )
)


# ------------------------------------------------------------
# 7. TOP PROFIT PRODUCTS
# ------------------------------------------------------------

top_profit_products = (
    dashboard_products
    .sort_values("adjusted_profit", ascending=False)
    .head(10)
)

print("\n" + "=" * 70)
print("TOP 10 PRODUCTS BY PROFIT")
print("=" * 70)

display(
    top_profit_products[
        [
            "product_name",
            "revenue",
            "adjusted_profit",
            "adjusted_margin_pct"
        ]
    ]
)


# ------------------------------------------------------------
# 8. PRODUCTS REQUIRING ATTENTION
# ------------------------------------------------------------

worst_products = (
    dashboard_products
    .sort_values("adjusted_profit", ascending=True)
    .head(10)
)

print("\n" + "=" * 70)
print("PRODUCTS REQUIRING ATTENTION")
print("=" * 70)

display(
    worst_products[
        [
            "product_name",
            "revenue",
            "waste_cost",
            "waste_pct",
            "adjusted_profit",
            "adjusted_margin_pct",
            "business_status"
        ]
    ]
)


# ------------------------------------------------------------
# 9. HIGH WASTE PRODUCTS
# ------------------------------------------------------------

high_waste_products = (
    dashboard_products
    .sort_values("waste_pct", ascending=False)
    .head(10)
)

print("\n" + "=" * 70)
print("HIGHEST WASTE PRODUCTS")
print("=" * 70)

display(
    high_waste_products[
        [
            "product_name",
            "waste_cost",
            "waste_pct",
            "adjusted_profit"
        ]
    ]
)


# ------------------------------------------------------------
# 10. SAVE DASHBOARD DATA
# ------------------------------------------------------------

output_folder = "dashboard_data"

os.makedirs(output_folder, exist_ok=True)


kpi_summary = pd.DataFrame({
    "metric": [
        "Total Revenue",
        "Adjusted Profit",
        "Waste Cost",
        "Units Sold",
        "Profit Margin"
    ],
    "value": [
        total_revenue,
        total_profit,
        total_waste_cost,
        total_units,
        overall_margin
    ]
})


kpi_summary.to_csv(
    f"{output_folder}/kpi_summary.csv",
    index=False
)

dashboard_products.to_csv(
    f"{output_folder}/product_dashboard.csv",
    index=False
)

top_profit_products.to_csv(
    f"{output_folder}/top_profit_products.csv",
    index=False
)

worst_products.to_csv(
    f"{output_folder}/products_requiring_attention.csv",
    index=False
)

high_waste_products.to_csv(
    f"{output_folder}/high_waste_products.csv",
    index=False
)


print("\n" + "=" * 70)
print("STAGE 6 DATA PREPARATION COMPLETE")
print("=" * 70)

print("\nFiles saved in:")
print(f"{os.path.abspath(output_folder)}")

STAGE 6 - DASHBOARD DATA PREPARATION


FileNotFoundError: Could not find 'profitability.csv'. Please enter the correct path to your profitability CSV.

In [4]:
# ============================================================
# CHECK WHAT FILES ARE AVAILABLE
# ============================================================

import os

print("Current working directory:")
print(os.getcwd())

print("\nFiles and folders available:")

for item in os.listdir():
    print(" -", item)

Current working directory:
c:\Users\user\Desktop\Bakery dashboard\dashboard

Files and folders available:
 - dashboard.ipynb
